# Elliptical Fourier contours · optimize physical *rotating-stack* non-normality

**Research objective:** Design a *single shared contour* used throughout a stack of directly adjacent patterned slices. The total device height is fixed at $H$, with $h=H/N$ and layer $j$ rotated by $\theta_j=2\pi j/N$. The optimization Maxwell solve contains **all layers and multiple reflections simultaneously**, in a single periodic lateral cell. This is not an isolated TiO₂ fin in air and not an EP search.

The objective in **cell 7** is entirely notebook-defined, taking the full complex 4×4 scattering matrix. It rewards meaningful eigenpolarization overlap, normalized splitting, non-scalar strength and measurable transmission of the **complete stack**; fabrication is evaluated using actual slice height. The old isolated-layer solver remains only as a control. The characterization notebooks independently test whether the *local transport generator* supports this physics.

The fitted T/L/H-inspired and Sobol contours are retained. Their exact rendered contours—not letter masks—are evaluated by TORCWA. The global S may be unitary in this lossless, subwavelength setting even when transmission is non-normal.

In [ ]:
# Shared engines live in ../core (see ../PROJECTS.md). Run with the notebook folder as cwd.
import sys; from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / 'core').resolve()))

In [ ]:
from pathlib import Path
from dataclasses import asdict
from functools import partial
import hashlib, json
import numpy as np
import matplotlib.pyplot as plt
import torch

import engine_ellipse_contour, engine_ellipse_rcwa, engine_ellipse_inverse
import engine_transmission_ep, engine_rcwa, engine_inverse
from engine_contour_checkpoint import make_manifest, manifest_id, read_candidate
from twofin import Config
from engine_rcwa import Settings as RCWASettings
from engine_ellipse_contour import (ContourRules, ContourDesign, contour_points,
    contour_fill, fabrication_cost, hard_mask_checks, periodic_margin,
    harmonic_modes, real_dtype, save_contour_svg, save_contour_gds)
from engine_ellipse_rcwa import solve_contour_torch
from engine_ellipse_inverse import (SearchSettings, FabricationSpread,
    perturbations, screen, optimize_start, evaluate, robust_report,
    optical_diagnostics_numpy, save_result, load_result,
    make_screening_initializations, SHAPE_FAMILIES)
from engine_transport_design import (TransportSetup, normalized_jones_metrics,
    solve_rotating_stack_torch, stack_fabrication_cost, solve_embedded_slice_torch)
from engine_transmission_ep import diagnostics, circular_blocks
from engine_multilayer_characterization import jones_metrics_numpy

OUT = Path('results/elliptical_contour_transmission_ep')
OUT.mkdir(parents=True,exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:',torch.__version__,'| device:',DEVICE)

## 1 · Physical assumptions and computational controls

No spacers: TiO₂ profiles touch the following layer directly along z, with the void regions filled by the configured uniform background medium. We initially use air (`background_index=1.0`) so the interfaces agree with the previous full-wave characterization, with silica substrate below. An *equal-ambient* thin-slice experiment is separate and must not be mistaken for a physical longitudinal propagation generator.

For improved adiabaticity, the **default total height is 633 nm**, held fixed as N varies; do not carry over the previous 105.5 nm total height. We train on N=6, then validate N=4, 8, 12, 20 without retraining. The total-height choice, per-layer thickness, and fabrication uncertainty are all saved in the checkpoint.

In [ ]:
# Screening -> order-based rerank -> differentiable Adam -> held-out N/thickness.
RUN_SMOKE_TEST = True
RUN_SEARCH = True
FORCE_SEARCH = False
RUN_HARD_VALIDATION = True
RUN_HELDOUT_ROBUSTNESS = True
RUN_EXPORT_GDS = False

TRAIN_N = 1
TOTAL_HEIGHT_UM = 0.633
STACK = TransportSetup(layers=TRAIN_N, background_index=1.0,
    bottom_index=1.46, top_index=1.0, rotation_sign=+1)
CHECK_N = (4, 6, 8, 12, 20)

smoke_settings = RCWASettings(order=2,raster=96,device=DEVICE,precision='complex128')
screen_settings = RCWASettings(order=3,raster=128,device=DEVICE,precision='complex64')
rerank_settings = RCWASettings(order=5,raster=192,device=DEVICE,precision='complex128')
coarse_settings = RCWASettings(order=4,raster=160,device=DEVICE,precision='complex64')
refine_settings = RCWASettings(order=6,raster=224,device=DEVICE,precision='complex128')
verify_orders = (12,14,16)
verify_raster = 256

cfg = Config(wavelength=0.633,period=0.400,n_sub=1.46,
             n_fin=2.584,n_fin_kappa=0.0)
rules = ContourRules(period=cfg.period,harmonics=16,angular_samples=1024,
    axis_ratio_bounds=(1.0,5.5),harmonic_amplitude=1.80,
    radius_bounds=(0.060,0.170),height_bounds=(TOTAL_HEIGHT_UM,TOTAL_HEIGHT_UM),
    cell_margin=0.020,perturbation_pad=0.008,
    min_periodic_gap_um=0.040,min_neck_um=0.050,
    min_tip_radius_um=0.012,preferred_aspect=10.0,maximum_aspect=16.0)
spread = FabricationSpread(etch_bias_um=.005,height_um=.010,contour_ripple_um=.003,
    layer_count=TRAIN_N,rotation_jitter_deg=1.0,registration_um=.003)
tuning = SearchSettings(n_screen=512,keep=10,
    coarse_iterations=150,refine_iterations=200,
    coarse_lr=.035,refine_lr=.008,coarse_patience=60,refine_patience=70,
    coarse_perturbations=1,refine_perturbations=2,
    fab_weight=.12,robust_weight=.20,seed=67,
    screening_nonnormality_bonus=0.,structured_fraction=.4,diversity_fraction=.4)

rules.validate(); STACK.validate()
print('N=',TRAIN_N,'H=',TOTAL_HEIGHT_UM,'um; thickness per layer=',
      1000*TOTAL_HEIGHT_UM/TRAIN_N,'nm')
print('Training includes full stack; optical loss accepts full complex S.')

## 2. Geometric anisotropy without mirror symmetry

The new `ContourDesign` variables are `[radius, height, axis_ratio, orientation, a_modes, b_modes]` in unconstrained optimizer coordinates. Axis ratio directly controls the elongated ellipse; the Fourier coefficients deform it in its **rotated local coordinates**. The cell-edge radius is capped conservatively to preserve clearance even after prescribed etch bias. Freeform asymmetry is encouraged by available degrees of freedom, **not enforced** by an arbitrary geometry penalty.

In [ ]:
example = ContourDesign(rules, device=DEVICE, dtype=real_dtype(refine_settings))
with torch.no_grad():
    example.raw[2] = -0.40  # Moderate, fabricable ellipse elongation
    example.raw[3] = 0.30  # Tilt the complete contour
    example.raw[4] = 0.50  # Odd harmonic: break inversion symmetry
    example.raw[5] = -0.30  # Third harmonic: break simple mirror symmetry
    example.raw[4+len(harmonic_modes(rules))] = 0.30
    p0 = example.values()
    xy = contour_points(p0,rules).detach().cpu().numpy()
    fill = contour_fill(p0,rules,256,hard=True).detach().cpu().numpy()
    fab,parts = fabrication_cost(p0,rules)
fig,axs=plt.subplots(1,2,figsize=(10,4))
axs[0].plot(*xy.T,'-',lw=2)
axs[0].set(xlim=(-.2,.2),ylim=(-.2,.2),aspect='equal',
           title='Rotated ellipse + asymmetric Fourier shape',xlabel='x (µm)',ylabel='y (µm)')
axs[1].imshow(fill.T,origin='lower',extent=[-.2,.2,-.2,.2],cmap='gray_r')
axs[1].set(title='Binary physical mask',xlabel='x (µm)',ylabel='y (µm)',aspect='equal')
fig.tight_layout();plt.show()
print('Base axis ratio:',float(p0['axis_ratio'].detach()),
      '| Orientation (deg):',np.rad2deg(float(p0['orientation'].detach())))
print('Fabrication proxies:',{k:round(float(v.detach()),5) for k,v in parts.items()})
print('Margin (µm):',periodic_margin(p0,rules).detach().cpu().numpy())
print('Hard-mask morphology:',hard_mask_checks(fill,rules))

## 2 · Editable optical objective (full 4×4 *stack* S)

All optical design choices live here. This objective **does not** target an EP. It targets moderate nonorthogonal transmission eigenvectors with resolvable eigenvalue splitting, non-scalar polarization response, and useful transmission. The numerical targets are hypotheses; if the loss remains high, we should report infeasibility rather than assert success. `normalized_jones_metrics` only computes invariants (including exact $\eta^2$ away from degeneracy). The notebook chooses the weights.

The separate `scattering_solver` callback below defines which physical Maxwell system is being optimized. It is applied during screening, Adam, and perturbation robustness. Changing it changes the scientific meaning of the checkpoint.

In [ ]:
OBJECTIVE_TAG = 'rotating_stack_nonorthogonal_v2'

# Edit these to explore observable polarization transformations.
TARGET_ETA = 0.65
MIN_GAP = 0.08
MIN_ANISOTROPY = 0.10
MIN_POWER = 0.60


def optical_loss(S):
    """Real scalar target of complete Maxwell stack scattering matrix S."""
    m = normalized_jones_metrics(S[2:4, :2])
    overlap = (m['eta2'] - TARGET_ETA**2).square()
    gap = torch.relu(MIN_GAP - m['gap']).square()
    anisotropy = torch.relu(MIN_ANISOTROPY - m['anisotropy']).square()
    transmission = torch.relu(MIN_POWER - m['mean_power']).square()
    loss = 2.0*overlap + 12.0*gap + 12.0*anisotropy + 5.0*transmission
    return loss, {**{k:v for k,v in m.items() if k!='delta'},
        'overlap_penalty':overlap,'gap_penalty':gap,
        'anisotropy_penalty':anisotropy,'power_penalty':transmission}


def scattering_solver(cfg, settings, p, rules, *, perturb=None,
                      edge_um=None, hard=False):
    """Differentiable, directly adjacent, rotated N-slice TORCWA stack."""
    return solve_rotating_stack_torch(cfg,settings,p,rules,setup=STACK,
        perturb=perturb,edge_um=edge_um,hard=hard)


def stack_fab(p, rules, *, perturb=None):
    return stack_fabrication_cost(p,rules,layers=TRAIN_N,perturb=perturb)

print('Objective:',OBJECTIVE_TAG,'| physical stack:',STACK)

## 3 · Shape initialization, gallery, and high-order reranking

H/T/L-inspired radial fits and Sobol shapes remain. We now screen their **complete rotating-stack** response. Before Adam, the best candidates are rerun at a higher Fourier order and ranked by both optical loss and sensitivity of the complex polarization response. A candidate with unstable near-degenerate eigenvectors should not advance because of a fortunate low-resolution score.

In [ ]:
initializations = make_screening_initializations(rules, tuning)
print(f'Generated {len(initializations)} starts:',
      {name: sum(s['family']==name for s in initializations)
       for name in (*SHAPE_FAMILIES, 'Sobol')})

# Show two of each letter-inspired family; add representative Sobol shapes.
preview_ids = [next(i for i,s in enumerate(initializations) if s['family']==f)
               for f in SHAPE_FAMILIES]
preview_ids += [next(i for i,s in enumerate(initializations)
                     if s['family']==f and i>preview_ids[j])
                for j,f in enumerate(SHAPE_FAMILIES[:4])]
preview_ids += [i for i,s in enumerate(initializations) if s['family']=='Sobol'][:4]
fig, axes = plt.subplots(4,4,figsize=(12,12))
for ax,idx in zip(axes.flat,preview_ids):
    item=initializations[idx]
    with torch.no_grad():
        preview_model=ContourDesign(rules,item['raw'],device='cpu',dtype=torch.float64)
        polygon=contour_points(preview_model.values(),rules).detach().numpy()
    ax.fill(polygon[:,0],polygon[:,1],color='#497cb3',alpha=.9)
    ax.plot(polygon[:,0],polygon[:,1],color='black',lw=.65)
    ax.set(aspect='equal',xlim=(-cfg.period/2,cfg.period/2),
           ylim=(-cfg.period/2,cfg.period/2),
           title=f"{item['family']} seed #{idx}" +
                 (f" | fit {item['fit_error']:.2f}" if item['fit_error'] is not None else ''))
    ax.grid(alpha=.13)
fig.suptitle('Actual bounded ellipse + Fourier starting contours',fontsize=15)
fig.tight_layout()
fig.savefig(OUT/'initialized_contours.png',dpi=170)
plt.show()


## 4 · Differentiability smoke test for the complete stack

The test computes gradients through all six patterned layers and checks one Fourier/ellipse parameter against finite differences. A passing test does not establish physical RCWA convergence.

In [ ]:
if RUN_SMOKE_TEST:
    test = ContourDesign(rules,device=DEVICE,dtype=real_dtype(smoke_settings))
    with torch.no_grad():
        test.raw[2] = .65
        test.raw[4] = .70
    def smoke_loss():
        p=test.values()
        loss,_,_=evaluate(cfg,smoke_settings,p,rules,loss_fn=optical_loss,
                          solver_fn=scattering_solver)
        fab,_=stack_fab(p,rules)
        return loss+tuning.fab_weight*fab
    test.raw.grad=None
    value=smoke_loss();value.backward()
    analytic=float(test.raw.grad[2].detach().cpu())
    eps_fd=0.002
    with torch.no_grad():
        saved=test.raw[2].clone()
        test.raw[2].copy_(saved+eps_fd);plus=float(smoke_loss())
        test.raw[2].copy_(saved-eps_fd);minus=float(smoke_loss())
        test.raw[2].copy_(saved)
    finite=(plus-minus)/(2*eps_fd)
    rel=abs(analytic-finite)/max(abs(analytic),abs(finite),1e-4)
    print(f'Stack gradient autodiff={analytic:.6g} finite difference={finite:.6g} relative={rel:.3g}')
    if not np.isfinite(rel) or rel>0.30:
        raise RuntimeError('Complete-stack gradient check failed')

## 5 · Screening → high-order rerank → Adam optimization

The same `optical_loss`, `scattering_solver`, and per-layer fabrication function are used in every stage. Every optimizer step logs the four new Jones invariants and individual objective components. Save and inspect intermediate results: a low loss does not establish geometric holonomy or longitudinally non-normal propagation.

In [ ]:
metadata=make_manifest(cfg=cfg,rules=rules,spread=spread,tuning=tuning,
    screen=screen_settings,coarse=coarse_settings,refine=refine_settings,
    verify_orders=verify_orders,verify_raster=verify_raster,kind='ellipse')
objective_bytes=(optical_loss.__code__.co_code+
                 repr(optical_loss.__code__.co_consts).encode())
metadata['objective']=dict(tag=OBJECTIVE_TAG,
    code_hash=hashlib.sha256(objective_bytes).hexdigest()[:16])
metadata['transport']=dict(kind='full_rotating_stack',train_N=TRAIN_N,
    total_height_um=TOTAL_HEIGHT_UM,setup=asdict(STACK),
    rerank=asdict(rerank_settings),check_N=list(CHECK_N),
    geometry='directly_adjacent_with_uniform_background')
run_id=manifest_id(metadata)
checkpoint=OUT/f'contour_transport_{run_id}.npz'
best=None
if checkpoint.exists() and not FORCE_SEARCH:
    saved=read_candidate(checkpoint,expected_kind='ellipse')
    if saved['metadata']!=metadata:raise ValueError('Cached configuration mismatch')
    best=load_result(checkpoint)
    print('Loaded transport candidate:',checkpoint)
elif RUN_SEARCH:
    starts,ranks,screen_records=screen(cfg,screen_settings,rules,tuning,
        initializations=initializations,loss_fn=optical_loss,
        solver_fn=scattering_solver,fab_fn=stack_fab,return_records=True,
        rerank_settings=rerank_settings,rerank_pool=18)
    (OUT/'transport_screening_results.json').write_text(
        json.dumps(screen_records,indent=2))
    print('High-order verified seeds:',[(r['family'],round(r['score'],4),
          round(r.get('high_order_drift',0),4)) for r in screen_records if r['selected']])
    coarse=[]
    for j,start in enumerate(starts,1):
        print(f'Coarse full-stack {j}/{len(starts)}')
        try:
            result=optimize_start(cfg,coarse_settings,rules,start,tuning,spread,
                iterations=tuning.coarse_iterations,lr=tuning.coarse_lr,
                patience=tuning.coarse_patience,n_perturbations=tuning.coarse_perturbations,
                loss_fn=optical_loss,solver_fn=scattering_solver,fab_fn=stack_fab)
            coarse.append(result)
        except (RuntimeError,ValueError) as exc: print('Coarse skip:',exc)
    if not coarse:raise RuntimeError('No successful full-stack coarse candidates')
    coarse.sort(key=lambda item:item['loss'])
    refined=[]
    for j,candidate in enumerate(coarse[:3],1):
        print(f'Refine full-stack {j}/3')
        try:
            result=optimize_start(cfg,refine_settings,rules,candidate['raw'],tuning,spread,
                iterations=tuning.refine_iterations,lr=tuning.refine_lr,
                patience=tuning.refine_patience,n_perturbations=tuning.refine_perturbations,
                loss_fn=optical_loss,solver_fn=scattering_solver,fab_fn=stack_fab)
            result['history']=candidate['history']+[
                {**r,'iteration':r['iteration']+len(candidate['history'])}
                for r in result['history']]
            refined.append(result)
        except (RuntimeError,ValueError) as exc: print('Refine skip:',exc)
    if not refined:raise RuntimeError('No successful refined candidates')
    best=min(refined,key=lambda item:item['loss'])
    save_result(checkpoint,best,metadata)
    print('Saved transport checkpoint:',checkpoint)
else:
    raise RuntimeError('Run search or supply an exact matching transport checkpoint')
print('Best total loss:',best['loss'])

## 6 · Optimized geometry and physical-stack training history

Plot the optimized mask, loss components, **full-stack** $\eta$, eigenvalue gap, non-scalar strength, and mean transmitted power. Check the fabricated per-layer thickness $H/N$; the saved raw `height` is now **total** height, not each layer's fin height.

In [ ]:
model=ContourDesign(rules,best['raw'],device=DEVICE,dtype=real_dtype(refine_settings))
with torch.no_grad():
    pp=model.values()
    coords=contour_points(pp,rules).detach().cpu().numpy()
    hard=contour_fill(pp,rules,384,hard=True).detach().cpu().numpy()
    soft=contour_fill(pp,rules,384).detach().cpu().numpy()
    fab,terms=stack_fab(pp,rules)
    height=float(pp['height'].cpu())
fig,axs=plt.subplots(1,3,figsize=(14,4))
axs[0].imshow(soft.T,origin='lower',extent=(-.2,.2,-.2,.2),vmin=0,vmax=1)
axs[0].set(title='Optimized contour (soft mask)')
axs[1].imshow(hard.T,origin='lower',extent=(-.2,.2,-.2,.2),cmap='gray_r')
axs[1].plot(coords[:,0],coords[:,1],color='tab:red',lw=.7)
axs[1].set(title=f'Binary profile, each slice {1000*height/TRAIN_N:.1f} nm')
axs[2].plot([r['loss'] for r in best['history']],label='total')
axs[2].plot([r['weighted_robustness'] for r in best['history']],label='weighted perturbations')
axs[2].set(yscale='symlog',title='Full-stack Adam trajectory',xlabel='step')
for ax in axs[:2]:ax.set_aspect('equal')
axs[2].legend();plt.tight_layout();plt.show()
fig,axs=plt.subplots(2,2,figsize=(10,7))
for ax,key in zip(axs.flat,('eta','gap','anisotropy','mean_power')):
    ax.plot([r.get('objective_'+key,np.nan) for r in best['history']])
    ax.set(title=key,xlabel='step');ax.grid(alpha=.3)
plt.tight_layout();plt.show()
print('Total H (um):',height,'physical layer height (um):',height/TRAIN_N)
print('Layer fabrication proxies:',{k:float(v.detach().cpu()) for k,v in terms.items()})
print('Hard-mask checks:',hard_mask_checks(hard,rules))

## 7 · Hard-mask full-stack convergence, multiple N and embedding controls

Different N at fixed H are *different geometries* and need not have the same eigenvectors. In addition to RCWA-order convergence at the trained N, test the same contour at other N, and compare against the old isolated whole-height fin. No finite-order result alone establishes a physical geometric phase.

In [ ]:
validation=[]
if RUN_HARD_VALIDATION:
    for order in verify_orders:
        setting=RCWASettings(order=order,raster=verify_raster,device=DEVICE,
                             precision='complex128')
        with torch.no_grad():
            p=ContourDesign(rules,best['raw'],device=DEVICE,dtype=real_dtype(setting)).values()
            S=scattering_solver(cfg,setting,p,rules,hard=True).detach().cpu().numpy()
        m=jones_metrics_numpy(S[2:4,:2]); delta=(np.nan if not validation
              else float(np.max(np.abs(S-validation[-1]['S']))))
        validation.append(dict(order=order,S=S,metrics=m,delta_previous=delta))
        print(f'Order {order}: eta={m["eta"]:.4f} gap={m["normalized_gap"]:.4f} '
              f'A={m["anisotropy"]:.4f} power={m["mean_power"]:.4f} max ΔS={delta:.4g}')
    S_final=validation[-1]['S']
else:
    S_final=best['S']
print('Trained stack metrics:',jones_metrics_numpy(S_final[2:4,:2]))
from engine_validate import display_matrix,plot_poincare
display_matrix(S_final,'Full physical rotating-stack S')
fig=plot_poincare(S_final,blocks=('T₂₁','T₁₂'));plt.show()

if RUN_HARD_VALIDATION:
    n_checks=[]
    setting=RCWASettings(order=verify_orders[-1],raster=verify_raster,
                        device=DEVICE,precision='complex128')
    with torch.no_grad():
        p=ContourDesign(rules,best['raw'],device=DEVICE,dtype=real_dtype(setting)).values()
        for N in CHECK_N:
            trial=TransportSetup(layers=N,background_index=STACK.background_index,
                bottom_index=STACK.bottom_index,top_index=STACK.top_index,rotation_sign=+1)
            S=solve_rotating_stack_torch(cfg,setting,p,rules,setup=trial,
                                        hard=True).detach().cpu().numpy()
            metrics=jones_metrics_numpy(S[2:4,:2]);n_checks.append(dict(N=N,S=S,**metrics))
            print('N',N,'h_nm',round(1000*height/N,2),metrics)
        S_single=solve_contour_torch(cfg,setting,p,rules,hard=True).detach().cpu().numpy()
    print('Original isolated whole-height fin:',jones_metrics_numpy(S_single[2:4,:2]))
    fig,axs=plt.subplots(2,2,figsize=(10,7))
    for ax,key in zip(axs.flat,('eta','normalized_gap','anisotropy','mean_power')):
        ax.plot([r['N'] for r in n_checks],[r[key] for r in n_checks],'o-')
        ax.set(xlabel='N',title='Full-stack '+key);ax.grid(alpha=.3)
    fig.tight_layout();plt.show()

In [ ]:
T21,T12=circular_blocks(S_final)
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,J,title in zip(axs[:2],(T21,T12),('T₂₁ circular eigenvalues','T₁₂ circular eigenvalues')):
    vals=np.linalg.eigvals(J)
    ax.scatter(vals.real,vals.imag,s=110)
    ax.plot(vals.real,vals.imag,'--',alpha=.6)
    ax.set(xlabel='Re(λ)',ylabel='Im(λ)',title=title)
    ax.grid(alpha=.2)
phase=np.rad2deg(np.angle(S_final))
phase=np.ma.masked_where(np.abs(S_final)<.02,phase)
im=axs[2].imshow(phase,cmap='twilight',vmin=-180,vmax=180)
axs[2].set(title='Complex S phase (°)',xlabel='Incoming channel',ylabel='Outgoing channel')
fig.colorbar(im,ax=axs[2]);fig.tight_layout();plt.show()

## 8 · Held-out process perturbations

Re-evaluate the **complete stack**, not a disconnected tall fin. Every perturbation is a new full-wave solve with total-thickness bias distributed across N slices. Report the numerical EP screen only as a legacy diagnostic: it is **not** our success criterion.

Perturbations also include independent ±1° orientation errors and 3 nm lateral registration errors for each of the N patterned slices. These errors act in the full-stack Maxwell solver, not only on the shape silhouette.

In [ ]:
if RUN_HELDOUT_ROBUSTNESS:
    test_settings=RCWASettings(order=6,raster=224,device=DEVICE,precision='complex128')
    checked=robust_report(cfg,test_settings,rules,best['raw'],spread,
            count=8,seed=8002,hard=True,loss_fn=optical_loss,
            solver_fn=scattering_solver)
    fig,axs=plt.subplots(1,3,figsize=(13,3.5))
    for ax,key in zip(axs,('T21_overlap','T21_gap','T21_nonnormality')):
        ax.plot([v['scenario'] for v in checked],[v[key] for v in checked],'o-')
        ax.set(xlabel='Perturbation scenario',title='Full-stack '+key)
        ax.grid(alpha=.25)
    plt.tight_layout();plt.show()
    print('Perturbation optical metrics:',checked)

## 9 · Export geometry, physical-stack S, and optimization manifest

The checkpoint includes the geometry rules, complete physical stack definition, source hashes and the notebook optical objective identity. `contour_transport_*.npz` is compatible with existing `read_candidate`. **Do not reinterpret the stored S as isolated single-fin scattering.**

In [ ]:
svg_path=save_contour_svg(OUT/'contour_transport_mask.svg',coords,cfg.period)
np.savez_compressed(OUT/f'transport_full_stack_{run_id}.npz',
    S=S_final,contour_xy_um=coords,hard_mask=hard,
    raw_params=np.asarray(best['raw'],float),total_height_um=height,
    physical_layer_height_um=height/TRAIN_N,training_layers=TRAIN_N,
    N_validation=np.array([r['N'] for r in n_checks]) if RUN_HARD_VALIDATION else [],
    validation_complex_S=np.array([r['S'] for r in n_checks]) if RUN_HARD_VALIDATION else [],
    hard_orders=np.array([v['order'] for v in validation]),
    convergence_delta=np.array([v['delta_previous'] for v in validation]))
if RUN_EXPORT_GDS:save_contour_gds(OUT/'contour_transport_mask.gds',coords,cfg.period)
print('Saved:',checkpoint,svg_path)

## 10 · Thickness dependence of embedded isolated slices (diagnostic only)

For the trained contour, compare **the entire stack** against equal-ambient thin slices at $h=H/N$. In a lossless local medium the forward generator may approach a normal/Hermitian operator even when a finite stack has non-normal transmission. The thin-slice Jones eigenvectors are not automatically eigenvectors of the true local $K(z)$; the multilayer notebook analyzes a two-way transfer-generator proxy and tests this distinction.

In [ ]:
if RUN_HARD_VALIDATION:
    setting=RCWASettings(order=verify_orders[-1],raster=verify_raster,
                         device=DEVICE,precision='complex128')
    slice_series=[]
    with torch.no_grad():
        q=ContourDesign(rules,best['raw'],device=DEVICE,dtype=real_dtype(setting)).values()
        for N in CHECK_N:
            S_slice=solve_embedded_slice_torch(cfg,setting,q,rules,N=N,
                background_index=STACK.background_index,hard=True).detach().cpu().numpy()
            m=jones_metrics_numpy(S_slice[2:4,:2])
            slice_series.append(dict(N=N,S=S_slice,**m))
            print(f'Equal-ambient slice H/{N}: eta={m["eta"]:.4f}, '
                  f'gap={m["normalized_gap"]:.4g}, A={m["anisotropy"]:.4g}')
    fig,axs=plt.subplots(1,3,figsize=(13,3.6))
    for ax,key in zip(axs,('eta','normalized_gap','anisotropy')):
        ax.plot([v['N'] for v in slice_series],[v[key] for v in slice_series],'.-',
                label='embedded slice')
        ax.plot([v['N'] for v in n_checks],[v[key] for v in n_checks],'o--',
                label='physical full stack')
        ax.set(xlabel='N',title=key);ax.grid(alpha=.3);ax.legend()
    plt.tight_layout();plt.show()
    np.savez_compressed(OUT/f'thickness_slices_{run_id}.npz',
         N=np.array([r['N'] for r in slice_series]),
         S=np.array([r['S'] for r in slice_series]))